In [ ]:
# ==================== 1. SETUP PATH & MOUNT DRIVE ====================
import os, sys, json, numpy as np, pandas as pd, matplotlib, warnings
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns, warnings, json, os, sys
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = '/content/drive/MyDrive'
# Struktur Flat: MyDrive/JURNAL-DATASET, JURNAL-PIPELINE, JURNAL-OUTPUTS
DATASET_ROOT = '/content/drive/MyDrive/JURNAL-DATASET'
PIPELINE_ROOT = '/content/drive/MyDrive/JURNAL-PIPELINE'
OUTPUTS_ROOT = '/content/drive/MyDrive/JURNAL-OUTPUTS'

paths = {
    'PREP': '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep',
    'LOG': '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/logs',
    'PRED': '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds',
    'FIG': '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/figures',
    'PREP': '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep',
}
os.makedirs('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/figures', exist_ok=True)

def save_fig(fig, fn, dpi=300):
    p = f'/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/figures/{fn}'
    fig.savefig(p, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"[SAVED] {p}")

import os, sys, json, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns, warnings
print("[INFO] Path & Env Siap.")

In [ ]:
import os

PRED_DIR = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/'

# Mapping: Nama File Lama -> Nama Baru (Sesuai Load Data)
rename_map = {
    'jurnal_final_hold_pred_s42.npy': 'jurnal_final_s42_hold_pred.npy',
    'jurnal_final_hold_pred_s43.npy': 'jurnal_final_s43_hold_pred.npy',
    'jurnal_final_hold_pred_s44.npy': 'jurnal_final_s44_hold_pred.npy',
}

print("[INFO] Cek & rename file seed...")

for old_name, new_name in {
    'jurnal_final_hold_pred_s42.npy': 'jurnal_final_s42_hold_pred.npy',
    'jurnal_final_hold_pred_s43.npy': 'jurnal_final_s43_hold_pred.npy',
    'jurnal_final_hold_pred_s44.npy': 'jurnal_final_s44_hold_pred.npy',
}.items():

    src = f'/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/{old_name}'
    dst = f'/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/{new_name}'

    if os.path.exists(src):
        if os.path.exists(dst):
            os.remove(src)  # Hapus file lama kalau file baru sudah ada
            print(f"[CLEAN] {old_name} dihapus (sudah ada {new_name})")
        else:
            os.rename(src, dst)
            print(f"[RENAMED] {old_name} → {new_name}")
    elif os.path.exists(f'/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/{new_name}'):
        print(f"[OK] {new_name} sudah benar.")
    else:
        print(f"[MISSING] {old_name} tidak ditemukan (akan di-handle di Load Data).")

print("[INFO] Rename cek selesai.")

In [ ]:
import numpy as np, os
from sklearn.metrics import confusion_matrix, f1_score

# Paths
PRED_DIR = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/'
PREP_DIR = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/'

# Load True Labels
y_true = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy')
print(f"[INFO] y_true: {np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy').shape}")

# Hybrid Pred
y_pred_hybrid = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy')

# Load Seeds for Ensemble
def load_seed(s):
    p = f'/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s{s}_hold_pred.npy'
    if os.path.exists(p): return np.load(p)
    print(f"[WARN] Seed {s} missing"); return None

s42 = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s42_hold_pred.npy') if os.path.exists('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s42_hold_pred.npy') else None
s43 = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s43_hold_pred.npy') if os.path.exists('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s43_hold_pred.npy') else None
s44 = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s44_hold_pred.npy') if os.path.exists('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s44_hold_pred.npy') else None

# Load Hybrid & True
y_true = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy')
y_pred_hybrid = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy')
y_true = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy') # Reload true

TH = np.array([0.30, 0.30, 0.30, 0.36])
CATEGORIES = ['Intrusive', 'Avoidance', 'Arousal', 'NegCogMood']
TH = np.array([0.30, 0.30, 0.30, 0.36])
CATEGORIES = ['Intrusive', 'Avoidance', 'Arousal', 'NegCogMood']

# Load Hybrid
y_pred_hybrid = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy')
y_true = np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy')

# Threshold
TH = np.array([0.30, 0.30, 0.30, 0.36])
CATEGORIES = ['Intrusive', 'Avoidance', 'Arousal', 'NegCogMood']

# Binarize
y_true_bin = (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy') >= np.array([0.30,0.30,0.30,0.36])).astype(int)
y_pred_hybrid_bin = (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy') >= np.array([0.30,0.30,0.30,0.36])).astype(int)

CATEGORIES = ['Intrusive', 'Avoidance', 'Arousal', 'NegCogMood']
TH = np.array([0.30, 0.30, 0.30, 0.36])

print("[INFO] Data Loaded. Siap untuk Fig 5, 6, 7.")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns, os

cms = [confusion_matrix(y_true_bin[:,i], (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy') >= np.array([0.3,0.3,0.3,0.36])).astype(int)[:,i]) for i in range(4)]
# Reload to be safe
y_true_bin = (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy') >= np.array([0.3,0.3,0.3,0.36])).astype(int)
y_pred_hybrid_bin = (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy') >= np.array([0.3,0.3,0.3,0.36])).astype(int)

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

cms = [confusion_matrix(y_true_bin[:,i], (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy') >= np.array([0.3,0.3,0.3,0.36])).astype(int)[:,i]) for i in range(4)]
# Better: use variables already loaded
y_true_bin = (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/y_te_v3.npy') >= np.array([0.3,0.3,0.3,0.36])).astype(int)
y_pred_hybrid_bin = (np.load('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_hybrid_hold_pred.npy') >= np.array([0.3,0.3,0.3,0.36])).astype(int)

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

cms = [confusion_matrix(y_true_bin[:,i], y_pred_hybrid_bin[:,i]) for i in range(4)]
fig, axes = plt.subplots(2, 2, figsize=(10,9))
axes = axes.flatten()
for i, (cm, cat) in enumerate(zip(cms, ['Intrusive','Avoidance','Arousal','NegCogMood'])):
    ax = axes[i]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Pred Neg','Pred Pos'], yticklabels=['True Neg','True Pos'],
                ax=ax, annot_kws={'size':14,'weight':'bold'}, linewidths=1, linecolor='white', square=True)
    tn, fp, fn, tp = cm.ravel()
    for j, (val, label) in enumerate(zip([tn,fp,fn,tp], ['TN','FP','FN','TP'])):
        ax.text(j%2+0.5, j//2+0.5, f'{label}\n{val}', ha='center', va='center', fontsize=12, fontweight='bold', color='black')
    ax.set_title(cat, fontsize=14, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted Label', fontsize=11); ax.set_ylabel('True Label', fontsize=11)

plt.suptitle('Confusion Matrix per Kategori TIR (Hybrid BERT-SVR, Test 271 Gold)', fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0,0,1,0.95])

# Save
import os
fig_path = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/figures/fig5_confusion_matrix_grid.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"[SAVED] Fig 5 saved.")

In [ ]:
import json, os, numpy as np, matplotlib.pyplot as plt

manifest_path = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/prep/manifest_assembli.json'
if os.path.exists(manifest_path):
    with open(manifest_path) as f: manifest = json.load(f)
    data = {k.capitalize(): manifest.get(k, {'gold':1573,'natural':1999,'batch3':577,'batch4':564,'batch5':600}[k]) for k in ['gold','natural','batch3','batch4','batch5']}
else:
    data = {'Gold':1573,'Natural':1999,'Batch3':577,'Batch4':564,'Batch5':600}

labels, sizes = list(data.keys()), list(data.values())
total = sum(data.values())
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']

fig, ax = plt.subplots(figsize=(7,7))
wedges, texts, autotexts = plt.pie(sizes, labels=list(data.keys()), autopct=lambda p: f'{p:.1f}%\n({int(p*sum(data.values())/100)})',
                                   colors=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd'],
                                   explode=(0.03,)*5, startangle=90, pctdistance=0.75, labeldistance=1.1,
                                  textprops={'fontsize':11,'fontweight':'bold'})
for at in autotexts: at.set_color('white'); at.set_fontweight('bold')
plt.title(f'Distribusi Data Latih (Total: {sum(data.values()):,})', fontsize=14, fontweight='bold', pad=20)
ax.legend(wedges, [f'{l}: {s:,} ({s/sum(data.values())*100:.1f}%)' for l,s in zip(data.keys(), data.values())],
          title='Sumber Data', loc='center left', bbox_to_anchor=(1,0,0.5,1))
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/figures/fig7_pie_dataset_distribution.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("[SAVED] Fig 7 Pie Chart")

In [ ]:
# CEK FILE SEED DULU
s42_p = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s42_hold_pred.npy'
s43_p = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s43_hold_pred.npy'
s44_p = '/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal/preds/jurnal_final_s44_hold_pred.npy'

has_seeds = all(os.path.exists(p) for p in [s42_p, s43_p, s44_p])

if not all(os.path.exists(p) for p in [s42_p, s43_p, s44_p]):
    print("⚠️ FILE SEED 42/43/44 BELUM ADA. Fig 6 akan pakai fallback (3 bar sama).")
    print("Jalankan NB09 & NB09a dulu untuk generate file .npy seed 42,43,44.")
else:
    print("✅ File seed 42,43,44 ditemukan. Generating Fig 6...")
    # Kode plotting Fig 6 di sini (sama seperti sebelumnya)
    # ... (paste code Fig 6 dari jawaban sebelumnya) ...